In [ ]:
import pandas as pd 
from tqdm import tqdm
import numpy as np
from sklearn import metrics

In [ ]:
def isNaN(num):
    return num != num

In [ ]:
def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True):
    """
    given a sklearn confusion matrix (cm), make a nice plot

    Arguments
    ---------
    cm:           confusion matrix from sklearn.metrics.confusion_matrix

    target_names: given classification classes such as [0, 1, 2]
                  the class names, for example: ['high', 'medium', 'low']

    title:        the text to display at the top of the matrix

    cmap:         the gradient of the values displayed from matplotlib.pyplot.cm
                  see http://matplotlib.org/examples/color/colormaps_reference.html
                  plt.get_cmap('jet') or plt.cm.Blues

    normalize:    If False, plot the raw numbers
                  If True, plot the proportions

    Usage
    -----
    plot_confusion_matrix(cm           = cm,                  # confusion matrix created by
                                                              # sklearn.metrics.confusion_matrix
                          normalize    = True,                # show proportions
                          target_names = y_labels_vals,       # list of names of the classes
                          title        = best_estimator_name) # title of graph

    Citiation
    ---------
    http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html

    """
    import matplotlib.pyplot as plt
    import numpy as np
    import itertools

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(8, 6),dpi=300)
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

# Read Files

In [ ]:
old_criteria_df = pd.read_excel(r'..\medical_data\study_criteria_table_label_V4.xlsx')

In [ ]:
old_criteria_df[~isNaN(old_criteria_df['True_Stage'])]

In [ ]:
old_criteria_df 

In [ ]:
criteria_df = pd.read_csv('..\medical_data\study_medical_table_V3.csv')

In [ ]:
criteria_df[criteria_df['FolderName'].str.contains('Oneill_Olivia_101917_2306.000')][['High_MMSE_Score']]

In [ ]:
criteria_df['True_Certainty'] = criteria_df['True_Certainty'].replace(np.nan, '', regex=True)
criteria_df['True_Stage'] = criteria_df['True_Stage'].replace(np.nan, '', regex=True)
criteria_df['True_Predicted'] = criteria_df['True_Stage'].replace(np.nan, '', regex=True)
criteria_df['True_Stage'] = criteria_df['True_Stage'].replace(np.nan, '', regex=True)
criteria_df['Note'] = criteria_df['Note'].replace(np.nan, '', regex=True)

In [ ]:
for index,row in old_criteria_df[~isNaN(old_criteria_df['True_Stage'])].iterrows():
    idx = criteria_df.index[criteria_df['FolderName'] ==row['FolderName']][0]
    if criteria_df.iloc[idx]['True_Stage'] == '':
        criteria_df.at[idx,'True_Stage'] = row['True_Stage']
        criteria_df.at[idx,'True_Certainty'] = row['True_Certainty']
        criteria_df.at[idx,'Note'] = row['Note']

In [ ]:
criteria_df[criteria_df['True_Stage'] !='']

In [ ]:
criteria_df

In [ ]:
#pd.read_excel(r'..\medical_data\study_criteria_table_label.xlsx')

# Update Criteria

In [ ]:
criteria_df['Predicted_Stage'] = None
criteria_df['Predicted_Disease'] = None
criteria_df['Predicted_Certainty'] = None

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    if row['Age'] < 50:
        criteria_df.at[index,'Predicted_Stage'] = 'Excluded'
    if row['Exclusion_Enc'] != 0:
        criteria_df.at[index,'Predicted_Stage'] = 'Excluded'

In [ ]:
criteria_df['Predicted_dT'] = np.nan

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    if row['Predicted_Stage'] =='Excluded':
        continue
    if row['CDR_Score'] >= 1:
        if (isNaN(row['Predicted_dT'])) | (abs(row['Predicted_dT']) > abs(row['CDR_dT'])): 
            criteria_df.at[index,'Predicted_dT'] = row['CDR_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'Dementia'
    if row['MMSE_Score'] < 25:
        if criteria_df.at[index,'Predicted_Stage']  == 'Dementia':
            criteria_df.at[index,'Predicted_Certainty'] = 'High'
        if (isNaN(criteria_df.at[index,'Predicted_dT'])) | (abs(criteria_df.at[index,'Predicted_dT']) > abs(row['MMSE_dT'])): 
            criteria_df.at[index,'Predicted_dT'] = row['MMSE_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'Dementia'
            if criteria_df.at[index,'Predicted_Certainty'] == None:
                criteria_df.at[index,'Predicted_Certainty'] = 'Low'
    if row['MoCA_Score'] < 20:
        if criteria_df.at[index,'Predicted_Stage'] == 'Dementia':
            criteria_df.at[index,'Predicted_Certainty'] = 'High'
        if (isNaN(criteria_df.at[index,'Predicted_dT'])) | (abs(criteria_df.at[index,'Predicted_dT']) > abs(row['MoCA_dT'])): 
            criteria_df.at[index,'Predicted_dT'] = row['MoCA_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'Dementia'
            if row['Predicted_Certainty'] == None:
                criteria_df.at[index,'Predicted_Certainty'] = 'Low'
    if (row['Dementia_Enc'] >= 1) & ((row['Symptomatic_Enc'] >= 1)|(row['Dementia_Med'])):
        if criteria_df.at[index,'Predicted_Stage'] == 'Dementia':
            criteria_df.at[index,'Predicted_Certainty'] = 'High'
        if (isNaN(criteria_df.at[index,'Predicted_dT'])) | (abs(criteria_df.at[index,'Predicted_dT']) > abs(row['Dementia_dT'])): 
            criteria_df.at[index,'Predicted_dT'] = row['Dementia_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'Dementia'
            if criteria_df.at[index,'Predicted_Certainty'] == None:
                criteria_df.at[index,'Predicted_Certainty'] = 'Low'

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    if row['Predicted_Stage'] =='Excluded':
        continue
    if row['CDR_Score'] == 0.5:
        if (isNaN(criteria_df.at[index,'Predicted_dT'])) | (abs(criteria_df.at[index,'Predicted_dT']) > abs(row['CDR_dT'])): 
            criteria_df.at[index,'Predicted_dT'] = row['CDR_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'MCI'
    if (row['MMSE_Score'] <= 27) & (row['MMSE_Score'] >= 25):
        if criteria_df.at[index,'Predicted_Stage']  == 'MCI':
            criteria_df.at[index,'Predicted_Certainty'] = 'High'
        if (isNaN(criteria_df.at[index,'Predicted_dT'])) | (abs(criteria_df.at[index,'Predicted_dT']) > abs(row['MMSE_dT'])): 
            criteria_df.at[index,'Predicted_dT'] = row['MMSE_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'MCI'
            if criteria_df.at[index,'Predicted_Certainty'] == None:
                criteria_df.at[index,'Predicted_Certainty'] = 'Low'
    if  (row['MoCA_Score'] >= 20)& (row['MoCA_Score'] <= 26):
        if criteria_df.at[index,'Predicted_Stage']  == 'MCI':
            criteria_df.at[index,'Predicted_Certainty'] = 'High'
        if (isNaN(criteria_df.at[index,'Predicted_dT'])) | (abs(criteria_df.at[index,'Predicted_dT']) > abs(row['MoCA_dT'])): 
            criteria_df.at[index,'Predicted_dT'] = row['MoCA_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'MCI'
            if criteria_df.at[index,'Predicted_Certainty'] == None:
                criteria_df.at[index,'Predicted_Certainty'] = 'Low'
    if (row['MCI_Enc'] >= 1) & ((row['Symptomatic_Enc'] >= 1)|(row['MCI_Med'])):
        if criteria_df.at[index,'Predicted_Stage']  == 'MCI':
            criteria_df.at[index,'Predicted_Certainty'] = 'High'
        if (isNaN(criteria_df.at[index,'Predicted_dT'])) | (abs(criteria_df.at[index,'Predicted_dT']) > abs(row['MCI_dT'])): 
            criteria_df.at[index,'Predicted_dT'] = row['MCI_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'MCI'
            if criteria_df.at[index,'Predicted_Certainty'] == None:
                criteria_df.at[index,'Predicted_Certainty'] = 'Low'

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    if row['Predicted_Stage'] is None:
        if row['Symptomatic_Enc'] >= 1:
            criteria_df.at[index,'Predicted_dT'] = row['Symptomatic_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'Symptomatic'
        if row['MCI_Enc'] >= 1:
            criteria_df.at[index,'Predicted_dT'] = row['Symptomatic_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'Symptomatic'
        if row['Dementia_Enc'] >= 1:
            criteria_df.at[index,'Predicted_dT'] = row['Symptomatic_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'Symptomatic'

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    if row['Predicted_Stage'] is None:
        criteria_df.at[index,'Predicted_Stage'] = 'No Dementia'

In [ ]:
for index, row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=True):
    Enc_num = 0
    dT = float('inf')
    predicted_disease = row['Predicted_Disease']
    for disease in ['AlzD','VaD','FTD','DLB','PD']:
        if not isNaN(row[disease + '_Enc']):
            if ((abs(row[disease + '_dT']) < abs(dT))):
                predicted_disease = disease
                Enc_num = row[disease + '_Enc']
                dT = row[disease + '_dT']
    if Enc_num != 0:
        criteria_df.at[index,'Predicted_Disease'] = predicted_disease
       

In [ ]:
criteria_df[criteria_df['FolderName'].str.contains('Oneill_Olivia_101917_2306.000')][['High_MMSE_Score']]

In [ ]:
#MCI Correction
for index, row in tqdm(criteria_df[(criteria_df['Predicted_Stage']=='Dementia')].iterrows(),total=criteria_df[criteria_df['Predicted_Stage']=='Dementia'].shape[0],position=0,leave=True):
    if isNaN(row['Low_CDR_Score']) & isNaN(row['High_MMSE_Score']) & isNaN(row['High_MoCA_Score']):
        continue
    if row['Low_CDR_Score'] <= 0.5:
        if (isNaN(row['Predicted_dT'])) | (row['Predicted_dT'] > row['Low_CDR_dT']): 
            criteria_df.at[index,'Predicted_dT'] = row['Low_CDR_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'MCI'
    if row['High_MMSE_Score'] > 25:
        if (isNaN(criteria_df.at[index,'Predicted_dT'])) | (criteria_df.at[index,'Predicted_dT'] > row['High_MMSE_dT']): 
            criteria_df.at[index,'Predicted_dT'] = row['High_MMSE_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'MCI'
    if row['High_MoCA_Score'] > 20:
        if (isNaN(criteria_df.at[index,'Predicted_dT'])) | (criteria_df.at[index,'Predicted_dT'] > row['High_MoCA_dT']): 
            criteria_df.at[index,'Predicted_dT'] = row['High_MoCA_dT']
            criteria_df.at[index,'Predicted_Stage'] = 'MCI'
    #if row['MCI_Enc'] >= 1:
    #    if criteria_df.at[index,'Predicted_dT'] > row['MCI_dT']:
    #        criteria_df.at[index,'Predicted_dT'] = row['MCI_dT']
    #        criteria_df.at[index,'Predicted_Stage'] = 'MCI'

In [ ]:
criteria_df[criteria_df['FolderName'].str.contains('Oneill_Olivia_101917_2306.000')]

In [ ]:
criteria_df[criteria_df['FolderName'].str.contains('Oneill_Olivia_101917_2306.000')][['True_Stage', 'Predicted_Stage',
        'Predicted_dT',
       'FolderName', 
      'MMSE_Score',
       'MMSE_Note', 'MMSE_dT', 'High_MMSE_Score', 'High_MMSE_Note',
       'High_MMSE_dT', 'High_MMSE_Score', 'MoCA_Score', 'MoCA_Note', 'MoCA_dT',
       'High_MoCA_Score', 'High_MoCA_Note', 'High_MoCA_dT']]

In [ ]:
print('Dementia Cases:',len(criteria_df[criteria_df['Predicted_Stage'] =='Dementia' ] ))
print('MCI Cases:',len(criteria_df[criteria_df['Predicted_Stage'] =='MCI' ] ))
print('Symptomatic Cases:',len(criteria_df[criteria_df['Predicted_Stage'] =='Symptomatic' ] ))
print('Non-Dementia Cases:',len(criteria_df[criteria_df['Predicted_Stage'] =='No Dementia' ] ))
print('Excluded Cases:',len(criteria_df[criteria_df['Predicted_Stage'] =='Excluded' ] ))

In [ ]:
len(criteria_df[criteria_df['Predicted_Stage'] =='Excluded' ]['PatientID'].unique())

In [ ]:
print('High Certainty Dementia Cases:',len(criteria_df[(criteria_df['Predicted_Stage'] =='Dementia' ) & (criteria_df['Predicted_Certainty'] =='High')]))
print('Low Certainty Dementia Cases:',len(criteria_df[(criteria_df['Predicted_Stage'] =='Dementia' ) & (criteria_df['Predicted_Certainty'] =='Low')] ))
print('High Certainty MCI Cases:',len(criteria_df[(criteria_df['Predicted_Stage'] =='MCI' ) & (criteria_df['Predicted_Certainty'] =='High')]))
print('Low Certainty MCI Cases:',len(criteria_df[(criteria_df['Predicted_Stage'] =='MCI' ) & (criteria_df['Predicted_Certainty'] =='Low')] ))

In [ ]:
print('AlzD:',len(criteria_df[criteria_df['Predicted_Disease'] == 'AlzD']))
print('VaD:',len(criteria_df[criteria_df['Predicted_Disease'] == 'VaD']))
print('FTD:',len(criteria_df[criteria_df['Predicted_Disease'] == 'FTD']))
print('DLB:',len(criteria_df[criteria_df['Predicted_Disease'] == 'DLB']))
print('PDD:',len(criteria_df[(criteria_df['Predicted_Disease'] == 'PD') & ((criteria_df['Predicted_Stage'] == 'MCI')|(criteria_df['Predicted_Stage'] == 'Dementia'))]))

# Compute Accuracy

In [ ]:
criteria_df[~isNaN(criteria_df['True_Stage'])] 

In [ ]:
print('Accuracy:',len(criteria_df[(criteria_df['True_Stage'] == criteria_df['Predicted_Stage'] ) & (criteria_df['Predicted_Stage'] != 'Excluded') ] )/len(criteria_df[(~isNaN(criteria_df['True_Stage']) & (criteria_df['Predicted_Stage'] != 'Excluded') )] ))

In [ ]:
criteria_df[(criteria_df['True_Stage'] == criteria_df['Predicted_Stage'] ) & (criteria_df['Predicted_Stage'] != 'Excluded') ] 

In [ ]:
criteria_df[(~isNaN(criteria_df['True_Stage']))  & (criteria_df['True_Stage'] != criteria_df['Predicted_Stage'] ) & (criteria_df['Predicted_Stage'] != 'Excluded') ] 

# Confusion Matrices

In [ ]:
#criteria_df = pd.read_excel('..\medical_data\study_criteria_table_label_V2.xlsx')

In [ ]:
criteria_df = criteria_df.fillna('')

In [ ]:
labels_df = criteria_df[(criteria_df['True_Stage']!='') & (criteria_df['Predicted_Stage'] != 'Excluded')]
labels_df 

In [ ]:
labels_df[labels_df['FolderName'].str.contains('Oneill_Olivia_101917_2306.000')]

In [ ]:
cm = metrics.confusion_matrix(labels_df['True_Stage'],labels_df['Predicted_Stage'],labels=['Dementia','MCI','Symptomatic','No Dementia'])

In [ ]:
plot_confusion_matrix(cm=cm,
                          target_names=['Dementia','MCI','Symptomatic','Nondementia'],
                          title='Confusion matrix',
                          cmap=None,
                          normalize=False)

In [ ]:
cm = metrics.confusion_matrix(labels_df[labels_df['Predicted_Certainty'] == 'High']['True_Stage'],labels_df[labels_df['Predicted_Certainty'] == 'High']['Predicted_Stage'],labels=['Dementia','MCI','Symptomatic','No Dementia'])

In [ ]:
plot_confusion_matrix(cm=cm,
                          target_names=['Dementia','MCI','Symptomatic','Nondementia'],
                          title='Confusion matrix with High Certainty',
                          cmap=None,normalize=False)

In [ ]:
cm = metrics.confusion_matrix(labels_df[labels_df['Predicted_Certainty'] == 'Low']['True_Stage'],labels_df[labels_df['Predicted_Certainty'] == 'Low']['Predicted_Stage'],labels=['Dementia','MCI','Symptomatic','No Dementia'])

In [ ]:
cm

In [ ]:
plot_confusion_matrix(cm=cm,
                          target_names=['Dementia','MCI','Symptomatic','Nondementia'],
                          title='Confusion matrix with Low Certainty',
                          cmap=None,normalize=False
                          )

In [ ]:
criteria_df

In [ ]:
criteria_df.to_excel('..\medical_data\study_criteria_table_label_V6.xlsx',index=False)